# Organize Playground

Interactively explore and test the `media-curator organize` workflow.

This notebook lets you:
1. Scan a directory with filters (extension, prefix, suffix, regex)
2. Preview the organize plan (dry-run) before touching any files
3. Execute when you're satisfied

**Nothing moves until you explicitly call `plan.execute(dry_run=False)`.**

In [1]:
# Add package sources to sys.path so imports work from the notebook
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
src_dirs = [
    "packages/core/src",
    "packages/fs-indexer/src",
    "packages/metadata/src",
    "packages/media-grouping/src",
    "packages/duplicate-detection/src",
    "packages/rename-engine/src",
    "packages/job-runner/src",
    "packages/ui-common/src",
    "apps/media-curator/src",
    "apps/disk-atlas/src",
    "apps/rename-studio/src",
    "apps/control-center/src",
]
for d in src_dirs:
    p = str(ROOT / d)
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Project root: {ROOT}")

Project root: d:\Users\sousa\dev\repos\tidy-forge


## 1. Configuration

Set the directory you want to organise and your preferences below.

In [2]:
# ---- EDIT THESE ----
try:
    from dotenv import load_dotenv
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "python-dotenv is required to load .env in the notebook. "
        "Install it with: pip install python-dotenv"
    ) from exc

load_dotenv(ROOT / ".env")

food_diary_dir = os.getenv("FOOD_DIARY_DIR")
if not food_diary_dir:
    raise ValueError(
        "FOOD_DIARY_DIR is not set. Add it to .env in the project root."
    )

SOURCE_DIR = Path(food_diary_dir)
DEST_DIR   = None                          # None = in-place; or Path("...") for a separate dest
DATE_FORMAT = "%Y%m"                       # subfolder format: "%Y%m" -> 202601, "%Y-%m" -> 2026-01
MODE       = "move"                        # "move" or "copy"
# --------------------

source = Path(SOURCE_DIR).resolve()
dest = Path(DEST_DIR).resolve() if DEST_DIR else source
print(f"Source:      {source}")
print(f"Destination: {dest}")
print(f"Format:      {DATE_FORMAT}  (e.g. subfolder name: '202601')")
print(f"Mode:        {MODE}")

Source:      F:\Media\Pictures\Health\Diet Journal\2025
Destination: F:\Media\Pictures\Health\Diet Journal\2025
Format:      %Y%m  (e.g. subfolder name: '202601')
Mode:        move


## 2. Scan with Filters

Customise which files to include. Set any filter to `None` to skip it.

In [3]:
import re

from tidyforge.fs_indexer import ExtensionFilter, PatternFilter, scan_directory

# ---- EDIT THESE (set to None to disable) ----
EXT_FILTER    = {".jpg", ".jpeg", ".png", ".heic"}  # or None for all media, or set() for all files
PREFIX_FILTER = None          # e.g. "IMG_"  — only files whose name starts with this
SUFFIX_FILTER = None          # e.g. "_edited" — only files whose stem ends with this
REGEX_FILTER  = None          # e.g. r"\d{4}" — regex matched against filename
# ----------------------------------------------

filters = []

if EXT_FILTER is not None:
    filters.append(ExtensionFilter(include=EXT_FILTER))
else:
    # Default: common media extensions
    from media_curator.cli import MEDIA_EXTENSIONS
    filters.append(ExtensionFilter(include=MEDIA_EXTENSIONS))

if PREFIX_FILTER is not None:
    filters.append(PatternFilter(pattern=f"^{re.escape(PREFIX_FILTER)}"))
if SUFFIX_FILTER is not None:
    filters.append(PatternFilter(pattern=f"{re.escape(SUFFIX_FILTER)}\\.[^.]*$"))
if REGEX_FILTER is not None:
    filters.append(PatternFilter(pattern=REGEX_FILTER))

print(f"Active filters: {len(filters)}")
for f in filters:
    print(f"  - {f}")

Active filters: 1
  - ExtensionFilter(include={'.heic', '.jpg', '.jpeg', '.png'}, exclude=set())


In [4]:
entries = list(scan_directory(source, filters=filters))
print(f"Found {len(entries)} files")

# Show the first few
for e in entries[:10]:
    print(f"  {e.name}  ({e.size_human})  modified: {e.modified:%Y-%m-%d %H:%M}")
if len(entries) > 10:
    print(f"  ... and {len(entries) - 10} more")

Found 3342 files
  20241013_181919.jpg  (2.5 MB)  modified: 2024-10-13 16:19
  20241014_034029.jpg  (2.8 MB)  modified: 2024-10-14 01:40
  20241014_034044.jpg  (2.6 MB)  modified: 2024-10-14 01:40
  20241014_034053.jpg  (2.6 MB)  modified: 2024-10-14 01:40
  20241014_034108.jpg  (2.3 MB)  modified: 2024-10-14 01:41
  20241014_034144.jpg  (2.7 MB)  modified: 2024-10-14 01:41
  20241014_073230.jpg  (2.0 MB)  modified: 2024-10-14 05:32
  20241014_104139.jpg  (2.6 MB)  modified: 2024-10-14 08:41
  20241014_171329.jpg  (2.9 MB)  modified: 2024-10-14 15:13
  20241014_171339.jpg  (1.8 MB)  modified: 2024-10-14 15:13
  ... and 3332 more


## 3. Build the Plan (Dry-Run Preview)

This shows exactly where each file would go. **No files are moved yet.**

In [5]:
from media_curator.organize import build_organize_plan

plan = build_organize_plan(entries, dest, date_format=DATE_FORMAT, mode=MODE)

# Validate
issues = plan.validate()
if issues:
    print("Validation issues:")
    for issue in issues:
        print(f"  {issue}")
else:
    print("No validation issues.")

print(f"\nPlan: {len(plan.actions):,} files to {MODE}")

Validation issues:
  Collision: 2 files target 'F:\Media\Pictures\Health\Diet Journal\2025\202507\20250729_053254.jpg'
  Collision: 2 files target 'F:\Media\Pictures\Health\Diet Journal\2025\202507\20250729_054045.jpg'
  Collision: 2 files target 'F:\Media\Pictures\Health\Diet Journal\2025\202507\20250729_091739.jpg'
  Collision: 2 files target 'F:\Media\Pictures\Health\Diet Journal\2025\202507\20250729_091745.jpg'
  Collision: 2 files target 'F:\Media\Pictures\Health\Diet Journal\2025\202507\20250729_123814.jpg'
  Collision: 2 files target 'F:\Media\Pictures\Health\Diet Journal\2025\202507\20250729_123817.jpg'
  Collision: 2 files target 'F:\Media\Pictures\Health\Diet Journal\2025\202507\20250729_124110.jpg'
  Collision: 2 files target 'F:\Media\Pictures\Health\Diet Journal\2025\202507\20250729_143724.jpg'
  Collision: 2 files target 'F:\Media\Pictures\Health\Diet Journal\2025\202507\20250729_183734.jpg'
  Collision: 2 files target 'F:\Media\Pictures\Health\Diet Journal\2025\202507\20

In [6]:
# Preview: show source -> destination for every file
# Group by destination subfolder for readability
from collections import defaultdict

groups = defaultdict(list)
for action in plan.actions:
    subfolder = action.destination.parent.name
    groups[subfolder].append(action)

for subfolder in sorted(groups):
    actions = groups[subfolder]
    print(f"\n{subfolder}/  ({len(actions)} files)")
    for a in actions[:5]:
        print(f"  {a.source.name}  ->  {subfolder}/{a.destination.name}")
    if len(actions) > 5:
        print(f"  ... and {len(actions) - 5} more")


202410/  (137 files)
  20241013_181919.jpg  ->  202410/20241013_181919.jpg
  20241014_034029.jpg  ->  202410/20241014_034029.jpg
  20241014_034044.jpg  ->  202410/20241014_034044.jpg
  20241014_034053.jpg  ->  202410/20241014_034053.jpg
  20241014_034108.jpg  ->  202410/20241014_034108.jpg
  ... and 132 more

202411/  (296 files)
  20241101_061609.jpg  ->  202411/20241101_061609.jpg
  20241101_062056.jpg  ->  202411/20241101_062056.jpg
  20241101_062102.jpg  ->  202411/20241101_062102.jpg
  20241101_062232.jpg  ->  202411/20241101_062232.jpg
  20241101_063710.jpg  ->  202411/20241101_063710.jpg
  ... and 291 more

202412/  (235 files)
  20241201_140943.jpg  ->  202412/20241201_140943.jpg
  20241201_175536.jpg  ->  202412/20241201_175536.jpg
  20241201_175830.jpg  ->  202412/20241201_175830.jpg
  20241202_062443.jpg  ->  202412/20241202_062443.jpg
  20241202_115633.jpg  ->  202412/20241202_115633.jpg
  ... and 230 more

202501/  (246 files)
  20250101_090458.jpg  ->  202501/20250101_09

In [7]:
# Dry-run: simulate execution and show the manifest
dry_manifest = plan.execute(dry_run=True)
print(dry_manifest.summary)
print()
for op in dry_manifest.operations[:10]:
    print(f"  {'OK' if op.success else 'FAIL'}  {op.message or op.error}")
if len(dry_manifest.operations) > 10:
    print(f"  ... and {len(dry_manifest.operations) - 10} more")

[DRY RUN] 3342 operations: 3342 succeeded, 0 failed

  OK  Would move '20241013_181919.jpg' -> 'F:\Media\Pictures\Health\Diet Journal\2025\202410\20241013_181919.jpg'
  OK  Would move '20241014_034029.jpg' -> 'F:\Media\Pictures\Health\Diet Journal\2025\202410\20241014_034029.jpg'
  OK  Would move '20241014_034044.jpg' -> 'F:\Media\Pictures\Health\Diet Journal\2025\202410\20241014_034044.jpg'
  OK  Would move '20241014_034053.jpg' -> 'F:\Media\Pictures\Health\Diet Journal\2025\202410\20241014_034053.jpg'
  OK  Would move '20241014_034108.jpg' -> 'F:\Media\Pictures\Health\Diet Journal\2025\202410\20241014_034108.jpg'
  OK  Would move '20241014_034144.jpg' -> 'F:\Media\Pictures\Health\Diet Journal\2025\202410\20241014_034144.jpg'
  OK  Would move '20241014_073230.jpg' -> 'F:\Media\Pictures\Health\Diet Journal\2025\202410\20241014_073230.jpg'
  OK  Would move '20241014_104139.jpg' -> 'F:\Media\Pictures\Health\Diet Journal\2025\202410\20241014_104139.jpg'
  OK  Would move '20241014_171329.j

## 4. Execute (for real)

**Only run the cell below when you're satisfied with the preview above.**

After the dry-run, the plan's actions are already marked "done", so we
rebuild a fresh plan before the real execution.

In [8]:
# Rebuild a fresh plan (dry-run consumed the previous one's statuses)
final_plan = build_organize_plan(entries, dest, date_format=DATE_FORMAT, mode=MODE)

# Uncomment the line below to actually move/copy the files:
manifest = final_plan.execute(dry_run=False)
print(manifest.summary)

[EXECUTED] 3342 operations: 3342 succeeded, 0 failed


In [9]:
# Optional: save the manifest to a JSON log
manifest.to_json(ROOT / "data" / "tmp" / "organize_manifest.json")
print("Manifest saved.")

Manifest saved.
